***<H1>LSTM Based</H1>***

***Subject dependent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def run_trial(seed_value=1737200047):
    # Set a random seed
    set_seed(seed_value)

    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels as tensor

    # Define the LSTM Model
    class LSTMModel(nn.Module):
        def __init__(self, input_size, hidden_size, output_size, num_layers=1):
            super(LSTMModel, self).__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
            self.fc = nn.Linear(hidden_size*2, output_size)

        def forward(self, x):
            lstm_out, _ = self.lstm(x)
            lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
            out = self.fc(lstm_out_last)
            return out

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Hyperparameters
    hidden_size = 128
    num_layers = 2
    batch_size = 8
    num_epochs = 200
    max_learning_rate = 5e-4  # Maximum learning rate
    min_learning_rate = 5e-5   # Minimum learning rate

    # Create Dataset
    audio_folder = 'subject_ordered_features/Training'
    dataset = AudioDataset(audio_folder)
    input_size = dataset.input_size  # Get input size dynamically
    output_size = len(dataset.label_map)  # Number of emotion classes

    # Create DataLoader
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

    # Use the entire dataset for training
    print(f'Training on {len(dataset)} samples.')

    # Initialize the model, loss function, and optimizer
    model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,  # Number of iterations for cosine cycle
        eta_min=min_learning_rate  # Minimum learning rate
    )

    loss_values = []

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        # Update learning rate
        scheduler.step()
        
        # Calculate epoch statistics
        epoch_loss = running_loss / len(train_loader)
        loss_values.append(epoch_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}')

        
        if (epoch + 1) % 5 == 0:
        # Save checkpoint every epoch
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': dataset.scaler
            }
            
            save_dir = f"saved_weights/subject_dependent/non_curriculum/lstm/random_seed_{seed_value}"
            os.makedirs(save_dir, exist_ok=True)
            torch.save(checkpoint, f"{save_dir}/epoch_{epoch + 1}_modelcheckpoint.pth")
            print(f"Checkpoint saved for epoch {epoch + 1}")

    print("Training complete!")

random_seed = 1737200047 # Vary random seed 

print(f"Starting Training with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Dependent - Testing***

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

random_seed = 1737200047 # Vary random seed 

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []

    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, scaler):
        self.audio_folder = audio_folder
        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()
        self.scaler = scaler  # Use the scaler fitted on the training data

        # Map emotion labels to indices
        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        file_paths = []
        labels = []
        for subject_id in os.listdir(self.audio_folder):
            subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
            if os.path.exists(subject_path):
                for filename in os.listdir(subject_path):
                    if filename.endswith('.csv'):
                        file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                        file_paths.append(file_path)
                        label = self.extract_label_from_filename(filename)
                        labels.append(label)
        return file_paths, labels

    def extract_label_from_filename(self, filename):
        # Extract label from filename (assuming format: subject_session_label.csv)
        label = filename.split('_')[2]
        return label

    def load_csv_data(self, file_path):
        # Load npy file and normalize the data using the scaler
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        # Determine the input size (number of features) from the first file
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Get normalized audio data and corresponding label
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)
    
    def get_counts(self):
        return len(self.file_paths)

# Define the LSTM model architecture
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        # LSTM layer
        lstm_out, _ = self.lstm(x)
        # Use the output of the last LSTM time step
        lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
        # Fully connected layer
        out = self.fc(lstm_out_last)
        return out

# Initialize a dictionary to store macro accuracies for each epoch across all trials
macro_accuracies = {
    25: [],
    50: [],
    100: [],
    150: [],
    200: []
}


print(f'Results for Seed: {random_seed}')
    
# Load testing data from specified folder
test_folder = 'subject_ordered_features/Testing'
test_dataset = TestDataset(test_folder, scaler=None) # Scaler will be loaded from each checkpoint

# Initialize lists to store accuracy for each epoch
epochs = [25, 50, 100, 150, 200]
test_accuracieslrs = []

# Store predictions and labels from the last epoch
final_labels = None
final_predictions = None
results = {
    'standard': [],
    'macro': [],
    'balanced': [],
    'total_count': 0,
}

# Evaluate the model for each epoch
for epoch in epochs:
    # Load checkpoint for the current epoch
    checkpoint_path = f"saved_weights/subject_dependent/non_curriculum/lstm/random_seed_{random_seed}/epoch_{epoch}_modelcheckpoint.pth"
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint for epoch {epoch} not found. Skipping.")
        continue

    # checkpoint = torch.load(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, weights_only=False)


    # Get model parameters from the checkpoint
    input_size = checkpoint['model_state_dict']['lstm.weight_ih_l0'].shape[1]  # Get input size from model weights
    hidden_size = 128  # Match with your training parameters
    num_layers = 2
    output_size = 6

    # Initialize the model and load the state dict
    model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load the scaler from the checkpoint
    scaler = checkpoint.get('scaler')
    test_dataset.scaler = scaler  # Update the scaler in the test dataset

    std_acc, macro_acc, bal_acc, report, labels, preds = evaluate_model(model, test_dataset, device)
    total_count = test_dataset.get_counts()
        
    # Store results
    results["standard"].append(std_acc)
    results["macro"].append(macro_acc)
    results["balanced"].append(bal_acc)
    results["total_count"] = total_count

    # Store macro accuracy for the epochs we're interested in
    if epoch in macro_accuracies:
        macro_accuracies[epoch].append(macro_acc)

    print(f"\nEpoch {epoch} Results:")
    print(f"- Standard Accuracy: {std_acc:.4f}")
    print(f"- Macro Accuracy: {macro_acc:.4f}")
    print(f"- Balanced Accuracy: {bal_acc:.4f}")
    print(f"Total test data points: {total_count}")
    print("\nClassification Report:")
    print(report)

    if epoch == epochs[-1]:
        final_labels = labels
        final_predictions = preds
        plot_confusion_matrix(
            final_labels, 
            final_predictions, 
            test_dataset.label_map,
            title='Confusion Matrix Non-Curriculum (LSTMs)'
        )

# Plot the accuracy metrics over epochs
plt.figure(figsize=(10, 5))
plt.plot(epochs, results["standard"], label='Standard Accuracy')
plt.plot(epochs, results["macro"], '--', label='Macro Accuracy')
plt.plot(epochs, results["balanced"], ':', label='Balanced Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Metrics Across Training')
plt.legend()
plt.grid(True)
plt.show()

# Plot the training loss if available
if 'loss_values' in checkpoint:
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(checkpoint['loss_values'])), checkpoint['loss_values'])
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Curve')
    plt.grid(True)
    plt.show()

***Subject Independent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def run_trial(seed_value=1737200047):
    # Set a random seed
    set_seed(seed_value)

    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels as tensor

    # Define the LSTM Model
    class LSTMModel(nn.Module):
        def __init__(self, input_size, hidden_size, output_size, num_layers=1):
            super(LSTMModel, self).__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
            self.fc = nn.Linear(hidden_size*2, output_size)

        def forward(self, x):
            lstm_out, _ = self.lstm(x)
            lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
            out = self.fc(lstm_out_last)
            return out

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Hyperparameters
    hidden_size = 128
    num_layers = 2
    batch_size = 8
    num_epochs = 200
    max_learning_rate = 5e-4  # Maximum learning rate
    min_learning_rate = 5e-5   # Minimum learning rate

    # Create Dataset
    audio_folder = 'subject_ordered_features/Training_SI'
    dataset = AudioDataset(audio_folder)
    input_size = dataset.input_size  # Get input size dynamically
    output_size = len(dataset.label_map)  # Number of emotion classes

    # Create DataLoader
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

    # Use the entire dataset for training
    print(f'Training on {len(dataset)} samples.')

    # Initialize the model, loss function, and optimizer
    model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,  # Number of iterations for cosine cycle
        eta_min=min_learning_rate  # Minimum learning rate
    )

    loss_values = []

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        # Update learning rate
        scheduler.step()
        
        # Calculate epoch statistics
        epoch_loss = running_loss / len(train_loader)
        loss_values.append(epoch_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}')

        
        if (epoch + 1) % 5 == 0:
        # Save checkpoint every epoch
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': dataset.scaler
            }
            
            save_dir = f"saved_weights/subject_independent/non_curriculum/lstm/random_seed_{seed_value}"
            os.makedirs(save_dir, exist_ok=True)
            torch.save(checkpoint, f"{save_dir}/epoch_{epoch + 1}_modelcheckpoint.pth")
            print(f"Checkpoint saved for epoch {epoch + 1}")

    print("Training complete!")

random_seed = 1737200047 # Vary random seed 

print(f"Starting Training with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Independent - Testing***

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

random_seed = 1737200047 # Vary random seed 

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []

    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, scaler):
        self.audio_folder = audio_folder
        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()
        self.scaler = scaler  # Use the scaler fitted on the training data

        # Map emotion labels to indices
        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        file_paths = []
        labels = []
        for subject_id in os.listdir(self.audio_folder):
            subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
            if os.path.exists(subject_path):
                for filename in os.listdir(subject_path):
                    if filename.endswith('.csv'):
                        file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                        file_paths.append(file_path)
                        label = self.extract_label_from_filename(filename)
                        labels.append(label)
        return file_paths, labels

    def extract_label_from_filename(self, filename):
        # Extract label from filename (assuming format: subject_session_label.csv)
        label = filename.split('_')[2]
        return label

    def load_csv_data(self, file_path):
        # Load npy file and normalize the data using the scaler
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        # Determine the input size (number of features) from the first file
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Get normalized audio data and corresponding label
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)
    
    def get_counts(self):
        return len(self.file_paths)

# Define the LSTM model architecture
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        # LSTM layer
        lstm_out, _ = self.lstm(x)
        # Use the output of the last LSTM time step
        lstm_out_last = lstm_out[:, -1, :]  # Take last time step for classification
        # Fully connected layer
        out = self.fc(lstm_out_last)
        return out

# Initialize a dictionary to store macro accuracies for each epoch across all trials
macro_accuracies = {
    25: [],
    50: [],
    100: [],
    150: [],
    200: []
}


print(f'Results for Seed: {random_seed}')
    
# Load testing data from specified folder
test_folder = 'subject_ordered_features/Testing_SI'
test_dataset = TestDataset(test_folder, scaler=None) # Scaler will be loaded from each checkpoint

# Initialize lists to store accuracy for each epoch
epochs = [25, 50, 100, 150, 200]
test_accuracieslrs = []

# Store predictions and labels from the last epoch
final_labels = None
final_predictions = None
results = {
    'standard': [],
    'macro': [],
    'balanced': [],
    'total_count': 0,
}

# Evaluate the model for each epoch
for epoch in epochs:
    # Load checkpoint for the current epoch
    checkpoint_path = f"saved_weights/subject_independent/non_curriculum/lstm/random_seed_{random_seed}/epoch_{epoch}_modelcheckpoint.pth"
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint for epoch {epoch} not found. Skipping.")
        continue

    # checkpoint = torch.load(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, weights_only=False)


    # Get model parameters from the checkpoint
    input_size = checkpoint['model_state_dict']['lstm.weight_ih_l0'].shape[1]  # Get input size from model weights
    hidden_size = 128  # Match with your training parameters
    num_layers = 2
    output_size = 6

    # Initialize the model and load the state dict
    model = LSTMModel(input_size, hidden_size, output_size, num_layers).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load the scaler from the checkpoint
    scaler = checkpoint.get('scaler')
    test_dataset.scaler = scaler  # Update the scaler in the test dataset

    std_acc, macro_acc, bal_acc, report, labels, preds = evaluate_model(model, test_dataset, device)
    total_count = test_dataset.get_counts()
        
    # Store results
    results["standard"].append(std_acc)
    results["macro"].append(macro_acc)
    results["balanced"].append(bal_acc)
    results["total_count"] = total_count

    # Store macro accuracy for the epochs we're interested in
    if epoch in macro_accuracies:
        macro_accuracies[epoch].append(macro_acc)

    print(f"\nEpoch {epoch} Results:")
    print(f"- Standard Accuracy: {std_acc:.4f}")
    print(f"- Macro Accuracy: {macro_acc:.4f}")
    print(f"- Balanced Accuracy: {bal_acc:.4f}")
    print(f"Total test data points: {total_count}")
    print("\nClassification Report:")
    print(report)

    if epoch == epochs[-1]:
        final_labels = labels
        final_predictions = preds
        plot_confusion_matrix(
            final_labels, 
            final_predictions, 
            test_dataset.label_map,
            title='Confusion Matrix Non-Curriculum (LSTMs)'
        )

# Plot the accuracy metrics over epochs
plt.figure(figsize=(10, 5))
plt.plot(epochs, results["standard"], label='Standard Accuracy')
plt.plot(epochs, results["macro"], '--', label='Macro Accuracy')
plt.plot(epochs, results["balanced"], ':', label='Balanced Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Metrics Across Training')
plt.legend()
plt.grid(True)
plt.show()

# Plot the training loss if available
if 'loss_values' in checkpoint:
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(checkpoint['loss_values'])), checkpoint['loss_values'])
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Curve')
    plt.grid(True)
    plt.show()

***<H1>Transformer Based</H1>***

***Subject Dependent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def run_trial(seed_value=1737200047):
    # Set a random seed
    set_seed(seed_value)

    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels as tensor

    # Positional Encoding for Transformer
    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=5000):
            super(PositionalEncoding, self).__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            pe = pe.unsqueeze(0)
            self.register_buffer('pe', pe)

        def forward(self, x):
            return x + self.pe[:, :x.size(1), :]

    # Transformer Encoder Model
    class TransformerEncoderModel(nn.Module):
        def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
            super(TransformerEncoderModel, self).__init__()
            self.d_model = d_model
            self.embedding = nn.Linear(input_size, d_model)
            self.pos_encoder = PositionalEncoding(d_model)
            encoder_layers = nn.TransformerEncoderLayer(
                d_model,
                nhead,
                dim_feedforward=d_model*4,
                dropout=dropout,
                batch_first=True
            )
            self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
            self.fc = nn.Linear(d_model, output_size)

        def forward(self, x):
            # x shape: (batch_size, seq_len, input_size)
            x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
            x = self.pos_encoder(x)
            x = self.transformer_encoder(x)
            x = x.mean(dim=1)  # Average over sequence length
            return self.fc(x)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Hyperparameters
    d_model = 128
    nhead = 4
    num_layers = 2
    batch_size = 8
    max_learning_rate = 5e-4
    min_learning_rate = 5e-5
    num_epochs = 400

    # Create Dataset
    audio_folder = 'subject_ordered_features/Training'
    dataset = AudioDataset(audio_folder)
    input_size = dataset.input_size  # Get input size dynamically
    output_size = len(dataset.label_map)  # Number of emotion classes

    # Create DataLoader
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

    # Use the entire dataset for training
    print(f'Training on {len(dataset)} samples.')

    # Initialize the model, loss function, and optimizer
    model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,  # Number of iterations for cosine cycle
        eta_min=min_learning_rate  # Minimum learning rate
    )

    loss_values = []

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        # Update learning rate
        scheduler.step()
        
        # Calculate epoch statistics
        epoch_loss = running_loss / len(train_loader)
        loss_values.append(epoch_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}')

        
        if (epoch + 1) % 5 == 0:
        # Save checkpoint every epoch
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': dataset.scaler
            }
            
            save_dir = f"saved_weights/subject_dependent/non_curriculum/transformer/random_seed_{seed_value}"
            os.makedirs(save_dir, exist_ok=True)
            torch.save(checkpoint, f"{save_dir}/epoch_{epoch + 1}_modelcheckpoint.pth")
            print(f"Checkpoint saved for epoch {epoch + 1}")

    print("Training complete!")

random_seed = 1737200047 # Vary random seed 

print(f"Starting Training with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Dependent - Testing***

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

random_seed = 1737200047 # Vary random seed 

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []

    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, scaler):
        self.audio_folder = audio_folder
        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()
        self.scaler = scaler  # Use the scaler fitted on the training data

        # Map emotion labels to indices
        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        file_paths = []
        labels = []
        for subject_id in os.listdir(self.audio_folder):
            subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
            if os.path.exists(subject_path):
                for filename in os.listdir(subject_path):
                    if filename.endswith('.csv'):
                        file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                        file_paths.append(file_path)
                        label = self.extract_label_from_filename(filename)
                        labels.append(label)
        return file_paths, labels

    def extract_label_from_filename(self, filename):
        # Extract label from filename (assuming format: subject_session_label.csv)
        label = filename.split('_')[2]
        return label

    def load_csv_data(self, file_path):
        # Load npy file and normalize the data using the scaler
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        # Determine the input size (number of features) from the first file
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Get normalized audio data and corresponding label
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)
    
    def get_counts(self):
        return len(self.file_paths)

# Positional Encoding for Transformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Transformer Encoder Model
class TransformerEncoderModel(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
        super(TransformerEncoderModel, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc = nn.Linear(d_model, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)  # Average over sequence length
        return self.fc(x)

# Initialize a dictionary to store macro accuracies for each epoch across all trials
macro_accuracies = {
    25: [],
    50: [],
    100: [],
    150: [],
    200: []
}


print(f'Results for Seed: {random_seed}')
    
# Load testing data from specified folder
test_folder = 'subject_ordered_features/Testing'
test_dataset = TestDataset(test_folder, scaler=None) # Scaler will be loaded from each checkpoint

# Initialize lists to store accuracy for each epoch
epochs = [25, 50, 100, 150, 200]
test_accuracieslrs = []

# Store predictions and labels from the last epoch
final_labels = None
final_predictions = None
results = {
    'standard': [],
    'macro': [],
    'balanced': [],
    'total_count': 0,
}

# Evaluate the model for each epoch
for epoch in epochs:
    # Load checkpoint for the current epoch
    checkpoint_path = f"saved_weights/subject_dependent/non_curriculum/transformer/random_seed_{random_seed}/epoch_{epoch}_modelcheckpoint.pth"
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint for epoch {epoch} not found. Skipping.")
        continue

    # checkpoint = torch.load(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, weights_only=False)


    # Get model parameters from the checkpoint
    input_size = checkpoint['model_state_dict']['embedding.weight'].shape[1]  # Get input size from model weights
    d_model = 128  # Match with your training parameters
    nhead = 4
    num_layers = 2
    output_size = 6

    # Initialize the model and load the state dict
    model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load the scaler from the checkpoint
    scaler = checkpoint.get('scaler')
    test_dataset.scaler = scaler  # Update the scaler in the test dataset

    std_acc, macro_acc, bal_acc, report, labels, preds = evaluate_model(model, test_dataset, device)
    total_count = test_dataset.get_counts()
        
    # Store results
    results["standard"].append(std_acc)
    results["macro"].append(macro_acc)
    results["balanced"].append(bal_acc)
    results["total_count"] = total_count

    # Store macro accuracy for the epochs we're interested in
    if epoch in macro_accuracies:
        macro_accuracies[epoch].append(macro_acc)

    print(f"\nEpoch {epoch} Results:")
    print(f"- Standard Accuracy: {std_acc:.4f}")
    print(f"- Macro Accuracy: {macro_acc:.4f}")
    print(f"- Balanced Accuracy: {bal_acc:.4f}")
    print(f"Total test data points: {total_count}")
    print("\nClassification Report:")
    print(report)

    if epoch == epochs[-1]:
        final_labels = labels
        final_predictions = preds
        plot_confusion_matrix(
            final_labels, 
            final_predictions, 
            test_dataset.label_map,
            title='Confusion Matrix Non-Curriculum (Transformers)'
        )

# Plot the accuracy metrics over epochs
plt.figure(figsize=(10, 5))
plt.plot(epochs, results["standard"], label='Standard Accuracy')
plt.plot(epochs, results["macro"], '--', label='Macro Accuracy')
plt.plot(epochs, results["balanced"], ':', label='Balanced Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Metrics Across Training')
plt.legend()
plt.grid(True)
plt.show()

# Plot the training loss if available
if 'loss_values' in checkpoint:
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(checkpoint['loss_values'])), checkpoint['loss_values'])
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Curve')
    plt.grid(True)
    plt.show()

***Subject Independent - Training***

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "7"  # For Python hash-based operations
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # For CuBLAS determinism (CUDA >= 10.2)


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", message="Deterministic behavior was enabled")

# Function to set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # If using CUDA
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def run_trial(seed_value=1737200047):
    # Set a random seed
    set_seed(seed_value)

    # Custom Dataset for handling the AU data from CSV files
    class AudioDataset(Dataset):
        def __init__(self, audio_folder):
            self.audio_folder = audio_folder
            self.file_paths, self.labels = self.load_data()
            self.input_size = self.get_input_size()

            self.scaler = StandardScaler()  # Initialize the scaler
            self.fit_scaler()  # Fit the scaler on the dataset

            self.label_map = {
                'ANG': 0,
                'DIS': 1,
                'FEA': 2,
                'HAP': 3,
                'NEU': 4,
                'SAD': 5
            }

        def load_data(self):
            file_paths = []
            labels = []
            for subject_id in os.listdir(self.audio_folder):
                subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
                if os.path.exists(subject_path):
                    for filename in os.listdir(subject_path):
                        if filename.endswith('.csv'):
                            file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                            file_paths.append(file_path)
                            label = self.extract_label_from_filename(filename)
                            labels.append(label)
            return file_paths, labels

        def extract_label_from_filename(self, filename):
            label = os.path.basename(filename).split('_')[2]
            return label

        def load_csv_data(self, file_path):
            df = np.load(file_path)
            audio_data = df[0]
            normalized_data = self.scaler.transform(audio_data)  # Normalize the data using the fitted scaler
            return torch.tensor(normalized_data, dtype=torch.float32)

        def get_input_size(self):
            first_file_path = self.file_paths[0] if self.file_paths else None
            if first_file_path:
                df = np.load(first_file_path)
                audio_columns = df.shape[-1]
                return audio_columns
            return 0

        def fit_scaler(self):
            # Collect all data for fitting the scaler
            data = []
            for file_path in self.file_paths:
                df = np.load(file_path)
                audio_data = df[0]
                data.append(audio_data)

            all_data = np.vstack(data)  # Stack all data vertically to fit the scaler
            self.scaler.fit(all_data)  # Fit the scaler on the entire dataset

        def __len__(self):
            return len(self.file_paths)

        def __getitem__(self, idx):
            file_path = self.file_paths[idx]
            label = self.labels[idx]
            audio_data = self.load_csv_data(file_path)
            label_index = self.label_map[label]
            return audio_data, label_index

    # Padding function to be used in the DataLoader collate_fn
    def pad_collate(batch):
        (xx, yy) = zip(*batch)
        xx_pad = pad_sequence(xx, batch_first=True)  # Pads all sequences to the same length
        return xx_pad, torch.tensor(yy, dtype=torch.long)  # Return labels as tensor

    # Positional Encoding for Transformer
    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=5000):
            super(PositionalEncoding, self).__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            pe = pe.unsqueeze(0)
            self.register_buffer('pe', pe)

        def forward(self, x):
            return x + self.pe[:, :x.size(1), :]

    # Transformer Encoder Model
    class TransformerEncoderModel(nn.Module):
        def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
            super(TransformerEncoderModel, self).__init__()
            self.d_model = d_model
            self.embedding = nn.Linear(input_size, d_model)
            self.pos_encoder = PositionalEncoding(d_model)
            encoder_layers = nn.TransformerEncoderLayer(
                d_model,
                nhead,
                dim_feedforward=d_model*4,
                dropout=dropout,
                batch_first=True
            )
            self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
            self.fc = nn.Linear(d_model, output_size)

        def forward(self, x):
            # x shape: (batch_size, seq_len, input_size)
            x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
            x = self.pos_encoder(x)
            x = self.transformer_encoder(x)
            x = x.mean(dim=1)  # Average over sequence length
            return self.fc(x)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Hyperparameters
    d_model = 128
    nhead = 4
    num_layers = 2
    batch_size = 8
    max_learning_rate = 5e-4
    min_learning_rate = 5e-5
    num_epochs = 400

    # Create Dataset
    audio_folder = 'subject_ordered_features/Training_SI'
    dataset = AudioDataset(audio_folder)
    input_size = dataset.input_size  # Get input size dynamically
    output_size = len(dataset.label_map)  # Number of emotion classes

    # Create DataLoader
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)

    # Use the entire dataset for training
    print(f'Training on {len(dataset)} samples.')

    # Initialize the model, loss function, and optimizer
    model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=max_learning_rate)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,  # Number of iterations for cosine cycle
        eta_min=min_learning_rate  # Minimum learning rate
    )

    loss_values = []

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        # Update learning rate
        scheduler.step()
        
        # Calculate epoch statistics
        epoch_loss = running_loss / len(train_loader)
        loss_values.append(epoch_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, LR: {current_lr:.2e}')

        
        if (epoch + 1) % 5 == 0:
        # Save checkpoint every epoch
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss_values': loss_values,
                'scaler': dataset.scaler
            }
            
            save_dir = f"saved_weights/subject_independent/non_curriculum/transformer/random_seed_{seed_value}"
            os.makedirs(save_dir, exist_ok=True)
            torch.save(checkpoint, f"{save_dir}/epoch_{epoch + 1}_modelcheckpoint.pth")
            print(f"Checkpoint saved for epoch {epoch + 1}")

    print("Training complete!")

random_seed = 1737200047 # Vary random seed 

print(f"Starting Training with random_seed:{random_seed}")
run_trial(seed_value=random_seed)

***Subject Independent - Testing***

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

random_seed = 1737200047 # Vary random seed 

def plot_confusion_matrix(all_labels, all_predictions, label_map, title='Confusion Matrix'):
    """
    Plot a confusion matrix using seaborn heatmap.
    
    Args:
    - all_labels: Array of true labels
    - all_predictions: Array of predicted labels
    - label_map: Dictionary mapping class names to indices
    - title: Title for the plot
    """
    # Get class names in order
    class_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    
    # Compute confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Normalize the confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Updated evaluation function with macro accuracy
def evaluate_model(model, test_dataset, device):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for idx in range(len(test_dataset)):
            inputs, labels = test_dataset[idx]
            inputs = inputs.unsqueeze(0).to(device)  # Add batch dimension and move to GPU
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            # Convert tensors to numpy arrays and collect
            all_labels.append(labels.cpu().numpy().item())  # Use .item() for scalar values
            all_predictions.append(predicted.cpu().numpy().item())

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    balanced_acc = balanced_accuracy_score(all_labels, all_predictions)
    
    # Calculate macro accuracy
    classes = np.unique(all_labels)
    per_class_acc = []

    for c in classes:
        mask = (all_labels == c)
        correct = np.sum(all_predictions[mask] == c)
        total = np.sum(mask)
        per_class_acc.append(correct / total if total > 0 else 0.0)
    macro_accuracy = np.mean(per_class_acc)

    report = classification_report(all_labels, all_predictions, 
                                target_names=list(test_dataset.label_map.keys()), 
                                labels=list(test_dataset.label_map.values()), 
                                digits=4, zero_division=1.0)

    return accuracy, macro_accuracy, balanced_acc, report, all_labels, all_predictions

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom Dataset for handling the AU data from CSV files for testing
class TestDataset(Dataset):
    def __init__(self, audio_folder, scaler):
        self.audio_folder = audio_folder
        self.file_paths, self.labels = self.load_data()
        self.input_size = self.get_input_size()
        self.scaler = scaler  # Use the scaler fitted on the training data

        # Map emotion labels to indices
        self.label_map = {
            'ANG': 0,
            'DIS': 1,
            'FEA': 2,
            'HAP': 3,
            'NEU': 4,
            'SAD': 5
        }

    def load_data(self):
        file_paths = []
        labels = []
        for subject_id in os.listdir(self.audio_folder):
            subject_path = os.path.join(self.audio_folder, subject_id, 'Audio')
            if os.path.exists(subject_path):
                for filename in os.listdir(subject_path):
                    if filename.endswith('.csv'):
                        file_path = os.path.join('hubert_features', filename.replace('csv','npy'))
                        file_paths.append(file_path)
                        label = self.extract_label_from_filename(filename)
                        labels.append(label)
        return file_paths, labels

    def extract_label_from_filename(self, filename):
        # Extract label from filename (assuming format: subject_session_label.csv)
        label = filename.split('_')[2]
        return label

    def load_csv_data(self, file_path):
        # Load npy file and normalize the data using the scaler
        df = np.load(file_path)
        audio_data = df[0]
        normalized_data = self.scaler.transform(audio_data)  # Normalize using the fitted scaler
        return torch.tensor(normalized_data, dtype=torch.float32)

    def get_input_size(self):
        # Determine the input size (number of features) from the first file
        first_file_path = self.file_paths[0] if self.file_paths else None
        if first_file_path:
            df = np.load(first_file_path)
            audio_columns = df.shape[-1]
            return audio_columns
        return 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Get normalized audio data and corresponding label
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        audio_data = self.load_csv_data(file_path)
        label_index = self.label_map[label]
        return audio_data, torch.tensor(label_index, dtype=torch.long)
    
    def get_counts(self):
        return len(self.file_paths)

# Positional Encoding for Transformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Transformer Encoder Model
class TransformerEncoderModel(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1):
        super(TransformerEncoderModel, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc = nn.Linear(d_model, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        x = self.embedding(x) * np.sqrt(self.d_model)  # (batch_size, seq_len, d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)  # Average over sequence length
        return self.fc(x)

# Initialize a dictionary to store macro accuracies for each epoch across all trials
macro_accuracies = {
    25: [],
    50: [],
    100: [],
    150: [],
    200: []
}


print(f'Results for Seed: {random_seed}')
    
# Load testing data from specified folder
test_folder = 'subject_ordered_features/Testing_SI'
test_dataset = TestDataset(test_folder, scaler=None) # Scaler will be loaded from each checkpoint

# Initialize lists to store accuracy for each epoch
epochs = [25, 50, 100, 150, 200]
test_accuracieslrs = []

# Store predictions and labels from the last epoch
final_labels = None
final_predictions = None
results = {
    'standard': [],
    'macro': [],
    'balanced': [],
    'total_count': 0,
}

# Evaluate the model for each epoch
for epoch in epochs:
    # Load checkpoint for the current epoch
    checkpoint_path = f"saved_weights/subject_independent/non_curriculum/transformer/random_seed_{random_seed}/epoch_{epoch}_modelcheckpoint.pth"
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint for epoch {epoch} not found. Skipping.")
        continue

    # checkpoint = torch.load(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, weights_only=False)


    # Get model parameters from the checkpoint
    input_size = checkpoint['model_state_dict']['embedding.weight'].shape[1]  # Get input size from model weights
    d_model = 128  # Match with your training parameters
    nhead = 4
    num_layers = 2
    output_size = 6

    # Initialize the model and load the state dict
    model = TransformerEncoderModel(input_size, d_model, nhead, num_layers, output_size).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load the scaler from the checkpoint
    scaler = checkpoint.get('scaler')
    test_dataset.scaler = scaler  # Update the scaler in the test dataset

    std_acc, macro_acc, bal_acc, report, labels, preds = evaluate_model(model, test_dataset, device)
    total_count = test_dataset.get_counts()
        
    # Store results
    results["standard"].append(std_acc)
    results["macro"].append(macro_acc)
    results["balanced"].append(bal_acc)
    results["total_count"] = total_count

    # Store macro accuracy for the epochs we're interested in
    if epoch in macro_accuracies:
        macro_accuracies[epoch].append(macro_acc)

    print(f"\nEpoch {epoch} Results:")
    print(f"- Standard Accuracy: {std_acc:.4f}")
    print(f"- Macro Accuracy: {macro_acc:.4f}")
    print(f"- Balanced Accuracy: {bal_acc:.4f}")
    print(f"Total test data points: {total_count}")
    print("\nClassification Report:")
    print(report)

    if epoch == epochs[-1]:
        final_labels = labels
        final_predictions = preds
        plot_confusion_matrix(
            final_labels, 
            final_predictions, 
            test_dataset.label_map,
            title='Confusion Matrix Non-Curriculum (Transformers)'
        )

# Plot the accuracy metrics over epochs
plt.figure(figsize=(10, 5))
plt.plot(epochs, results["standard"], label='Standard Accuracy')
plt.plot(epochs, results["macro"], '--', label='Macro Accuracy')
plt.plot(epochs, results["balanced"], ':', label='Balanced Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Metrics Across Training')
plt.legend()
plt.grid(True)
plt.show()

# Plot the training loss if available
if 'loss_values' in checkpoint:
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(checkpoint['loss_values'])), checkpoint['loss_values'])
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Curve')
    plt.grid(True)
    plt.show()